In [83]:
import sqlite3
from typing import Annotated, Literal
from langchain_core.messages import AIMessage
from pydantic import BaseModel, Field
from typing_extensions import TypedDict
from langgraph.graph import END, StateGraph, START
from langgraph.graph.message import AnyMessage, add_messages
from typing import Any
from langchain_core.messages import ToolMessage
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks
from langgraph.prebuilt import ToolNode

In [51]:
connection = sqlite3.connect('mydb.db')

In [52]:
connection

In [53]:
table_creation_query1 = """
                        CREATE TABLE IF NOT EXISTS employees
                        (
                            id
                            INTEGER
                            NOT
                            NULL
                            PRIMARY
                            KEY,
                            first_name
                            TEXT
                            NOT
                            NULL,
                            last_name
                            TEXT
                            NOT
                            NULL,
                            email
                            TEXT
                            UNIQUE
                            NOT
                            NULL,
                            hire_date
                            TEXT
                            NOT
                            NULL,
                            salary
                            REAL
                            NOT
                            NULL
                        ); \
                        """

In [54]:
table_creation_query2 = """
                        CREATE TABLE IF NOT EXISTS customers
                        (
                            id
                            INTEGER
                            NOT
                            NULL
                            PRIMARY
                            KEY,
                            first_name
                            TEXT
                            NOT
                            NULL,
                            last_name
                            TEXT
                            NOT
                            NULL,
                            email
                            TEXT
                            UNIQUE
                            NOT
                            NULL,
                            phone
                            TEXT
                        ); \
                        """

In [55]:
table_creation_query3 = """
                        CREATE TABLE IF NOT EXISTS orders
                        (
                            id
                            INTEGER
                            NOT
                            NULL
                            PRIMARY
                            KEY,
                            customer_id
                            INTEGER
                            NOT
                            NULL,
                            order_date
                            TEXT
                            NOT
                            NULL,
                            amount
                            REAL
                            NOT
                            NULL,
                            FOREIGN
                            KEY
                        (
                            customer_id
                        ) REFERENCES customers
                        (
                            id
                        )
                            ); \
                        """

In [56]:
cursor = connection.cursor()

In [57]:
cursor.execute(table_creation_query1)
cursor.execute(table_creation_query2)
cursor.execute(table_creation_query3)

In [58]:
employee_data = [
    (1, 'John', 'Doe', 'john.doe@company.com', '2020-01-15', 75000.00),
    (2, 'Jane', 'Smith', 'jane.smith@company.com', '2019-03-20', 82000.00),
    (3, 'Robert', 'Johnson', 'robert.johnson@company.com', '2021-06-10', 68000.00),
    (4, 'Maria', 'Garcia', 'maria.garcia@company.com', '2020-11-01', 71000.00),
    (5, 'David', 'Williams', 'david.williams@company.com', '2018-09-05', 95000.00)
]

cursor.executemany("""
                   INSERT
                   OR IGNORE INTO employees (id, first_name, last_name, email, hire_date, salary)
    VALUES (?, ?, ?, ?, ?, ?)
                   """, employee_data)

customer_data = [
    (1, 'Alice', 'Brown', 'alice.brown@email.com', '555-0101'),
    (2, 'Bob', 'Miller', 'bob.miller@email.com', '555-0102'),
    (3, 'Carol', 'Davis', 'carol.davis@email.com', '555-0103'),
    (4, 'Eve', 'Wilson', 'eve.wilson@email.com', None),
    (5, 'Frank', 'Moore', 'frank.moore@email.com', '555-0105')
]

cursor.executemany("""
                   INSERT
                   OR IGNORE INTO customers (id, first_name, last_name, email, phone)
    VALUES (?, ?, ?, ?, ?)
                   """, customer_data)

order_data = [
    (1, 1, '2023-01-15', 249.99),
    (2, 1, '2023-02-20', 499.50),
    (3, 2, '2023-01-28', 125.75),
    (4, 3, '2023-03-05', 899.00),
    (5, 2, '2023-03-12', 350.25),
    (6, 4, '2023-02-14', 175.50),
    (7, 5, '2023-01-30', 675.00),
    (8, 3, '2023-03-18', 220.80),
    (9, 1, '2023-03-22', 129.99),
    (10, 5, '2023-02-28', 450.00)
]

cursor.executemany("""
                   INSERT
                   OR IGNORE INTO orders (id, customer_id, order_date, amount)
    VALUES (?, ?, ?, ?)
                   """, order_data)

connection.commit()

In [59]:
cursor.execute("select * from customers")

In [60]:
for row in cursor.fetchall():
    print(row)

(1, 'Alice', 'Brown', 'alice.brown@email.com', '555-0101')
(2, 'Bob', 'Miller', 'bob.miller@email.com', '555-0102')
(3, 'Carol', 'Davis', 'carol.davis@email.com', '555-0103')
(4, 'Eve', 'Wilson', 'eve.wilson@email.com', None)
(5, 'Frank', 'Moore', 'frank.moore@email.com', '555-0105')


In [61]:
from langchain_community.utilities import SQLDatabase

In [62]:
db = SQLDatabase.from_uri("sqlite:///mydb.db")

In [63]:
db.dialect

'sqlite'

In [64]:
db.get_usable_table_names()

['customers', 'employees', 'orders']

In [65]:
import os

os.environ["GROQ_API_KEY"] = ".."

In [66]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b")

In [67]:
llm.invoke("Does Groq charge money?")

AIMessage(content='**Short answer:**  \nYes—Groq is a commercial business that charges for its products and services.  \n\n---\n\n## What Groq offers and how pricing works\n\n| Product / Service | Typical pricing model | Notes |\n|-------------------|-----------------------|-------|\n| **Hardware accelerators (e.g., Groq Tensor Streaming Processor)** | **Capital expenditure** – you purchase the chip or a server that contains it. Prices vary by model, quantity, and configuration. | Often sold through distributors or directly to enterprises; bulk discounts apply. |\n| **Groq Cloud (or “Groq Cloud Platform”)** | **Pay‑as‑you‑go / subscription** – you pay for compute time, storage, and data transfer. | Pricing is tier‑based; some customers get a free trial or a limited free tier (e.g., a few free GPU‑hours per month). |\n| **Software / SDK** | Usually free to download and use; some advanced tooling or enterprise support may cost. | The core SDK is open‑source, but enterprise support contra

In [68]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

In [69]:
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [70]:
tools = toolkit.get_tools()

In [71]:
for tool in tools:
    print(tool.name)

sql_db_query
sql_db_schema
sql_db_list_tables
sql_db_query_checker


In [72]:
list_tables_tool = next((tool for tool in tools if tool.name == 'sql_db_list_tables'), None)

In [73]:
list_tables_tool

ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001F33C6F7650>)

In [74]:
get_shema_tool = next((tool for tool in tools if tool.name == 'sql_db_schema'), None)

In [75]:
get_shema_tool

InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x000001F33C6F7650>)

In [76]:
list_tables_tool.invoke("")

'customers, employees, orders'

In [77]:
print(get_shema_tool.invoke("customers, employees, orders"))


CREATE TABLE customers (
	id INTEGER NOT NULL, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	phone TEXT, 
	PRIMARY KEY (id), 
	UNIQUE (email)
)

/*
3 rows from customers table:
id	first_name	last_name	email	phone
1	Alice	Brown	alice.brown@email.com	555-0101
2	Bob	Miller	bob.miller@email.com	555-0102
3	Carol	Davis	carol.davis@email.com	555-0103
*/


CREATE TABLE employees (
	id INTEGER NOT NULL, 
	first_name TEXT NOT NULL, 
	last_name TEXT NOT NULL, 
	email TEXT NOT NULL, 
	hire_date TEXT NOT NULL, 
	salary REAL NOT NULL, 
	PRIMARY KEY (id), 
	UNIQUE (email)
)

/*
3 rows from employees table:
id	first_name	last_name	email	hire_date	salary
1	John	Doe	john.doe@company.com	2020-01-15	75000.0
2	Jane	Smith	jane.smith@company.com	2019-03-20	82000.0
3	Robert	Johnson	robert.johnson@company.com	2021-06-10	68000.0
*/


CREATE TABLE orders (
	id INTEGER NOT NULL, 
	customer_id INTEGER NOT NULL, 
	order_date TEXT NOT NULL, 
	amount REAL NOT NULL, 
	PRIMARY KEY (id

In [78]:
from langchain_core.tools import tool


@tool
def db_query_tool(query: str) -> str:
    """
    Executes a query and returns the result.
    If the query is invalid or returns no result, an error message is returned.
    In case of an error, the user is adviced to rewrite query and try again.
    :param query:
    :return:
    """
    result = db.run_no_throw(query)
    if result is None:
        return "Error: Query failed. Please rewrite."
    return result

In [79]:
db_query_tool.invoke("SELECT * FROM customers")

"[(1, 'Alice', 'Brown', 'alice.brown@email.com', '555-0101'), (2, 'Bob', 'Miller', 'bob.miller@email.com', '555-0102'), (3, 'Carol', 'Davis', 'carol.davis@email.com', '555-0103'), (4, 'Eve', 'Wilson', 'eve.wilson@email.com', None), (5, 'Frank', 'Moore', 'frank.moore@email.com', '555-0105')]"

In [80]:
from langchain_core.prompts import ChatPromptTemplate

query_check_system = """You are a SQL expert with a strong attention to detail.
Double check the SQLite query for common mistakes, including:

- Using NOT IN with NULL values
- Using UNION when UNION ALL should have been used
- Using BETWEEN for exclusive ranges
- Data type mismatch in predicates
- Properly quoting identifiers
- Using the correct number of arguments for functions
- Casting to the correct data type
- Using the proper columns for joins

If there are any of the above mistakes, rewrite the query. If there are no mistakes, just reproduce the original query.

You will call the appropriate tool to execute the query after running this check."""

query_check_prompt = ChatPromptTemplate.from_messages([("system", query_check_system), ("placeholder", "{messages}")])

query_check = query_check_prompt | llm.bind_tools([db_query_tool])

In [81]:
print(query_check.invoke({"messages": [("user", "SELECT * FROM customers;")]}))

content='' additional_kwargs={'reasoning_content': 'We need to double-check for mistakes. Query: "SELECT * FROM customers;" This is simple. No issues: no NOT IN, no UNION, no BETWEEN, no data type mismatch, quoting identifiers: customers is likely a valid identifier; no function usage. So no mistakes. So we just reproduce the original query. Then we call the tool to execute it.', 'tool_calls': [{'id': 'fc_8b618959-b9fa-4b0f-ab04-56ca6fe4934d', 'function': {'arguments': '{"query":"SELECT * FROM customers;"}', 'name': 'db_query_tool'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 102, 'prompt_tokens': 299, 'total_tokens': 401, 'completion_time': 0.104916001, 'completion_tokens_details': {'reasoning_tokens': 74}, 'prompt_time': 0.016972011, 'prompt_tokens_details': None, 'queue_time': 0.035446641, 'total_time': 0.121888012}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_9340e7d14d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls'

In [84]:
class SubmitFinalAnswer(BaseModel):
    """Submit the final answer to the user based on the query results."""
    final_answer: str = Field(..., description="The final answer to the user.")


query_gen_system = """You are a SQL expert with a strong attention to detail.

Given an input question, output a syntactically correct SQLite query to run, then look at the results of the query and return the answer.

DO NOT call any tool besides SubmitFinalAnswer to submit the final answer.

When generating the query:

Output the SQL query that answers the input question without a tool call.

Unless the user specifies a specific number of examples they wish to obtain, always limit your query to at most 5 results.
You can order the results by a relevant column to return the most interesting examples in the database.
Never query for all the columns from a specific table, only ask for the relevant columns given the question.

If you get an error while executing a query, rewrite the query and try again.

If you get an empty result set, you should try to rewrite the query to get a non-empty result set.
NEVER make stuff up if you don't have enough information to answer the query... just say you don't have enough information.

If you have enough information to answer the input question, simply invoke the appropriate tool to submit the final answer to the user.

DO NOT make any DML statements (INSERT, UPDATE, DELETE, DROP etc.) to the database. Do not return any sql query except answer."""

query_gen_prompt = ChatPromptTemplate.from_messages([("system", query_gen_system), ("placeholder", "{messages}")])

query_gen = query_check_prompt | llm.bind_tools([SubmitFinalAnswer])

In [85]:
class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [86]:
def first_tool_call(state: State):
    pass

In [87]:
def handle_tool_error(state: State):
    pass

In [89]:
from typing import Dict

def create_tool_node_with_fallback(tools: list) -> RunnableWithFallbacks[Any, Dict]:
    pass

In [90]:
def query_gen_node(state: State):
    pass

In [91]:
def should_continue(state: State):
    pass

In [92]:
def model_check_query(state: State):
    pass

In [93]:
workflow = StateGraph(State)

In [94]:
workflow.add_node()
workflow.add_node()
workflow.add_node()
workflow.add_node()
workflow.add_node()
workflow.add_node()
workflow.add_node()

TypeError: StateGraph.add_node() missing 1 required positional argument: 'node'

In [ ]:
workflow.add_edge()
workflow.add_edge()
workflow.add_edge()
workflow.add_edge()
workflow.add_edge()
workflow.add_edge()
workflow.add_edge()
workflow.add_edge()
workflow.add_edge()

In [96]:
app = workflow.compile()

ValueError: Graph must have an entrypoint: add at least one edge from START to another node

In [97]:
from IPython.display import Image, display
from langchain_core.runnables.graph import MermaidDrawMethod

display(
    Image(
        app.get_graph().draw_mermaid_png(
            draw_method=MermaidDrawMethod.API
        )
    )
)

NameError: name 'app' is not defined

In [95]:
app.invoke("")

NameError: name 'app' is not defined

In [ ]:
app.stream()